# Notebook 6: Reliable delivery, journaling, and checkpoints

This notebook demonstrates the operational side of the repo: journaling memory mutations, buffering outbound work in a retryable outbox, and saving checkpoints so an agent can resume safely.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def show(data):
    if isinstance(data, str):
        print(data)
    else:
        print(json.dumps(data, indent=2, default=str))


In [ ]:
from src.active_memory.checkpoint import CheckpointManager
from src.journal.models import JournalEvent
from src.journal.outbox import RetryableOutbox

memory_root = REPO_ROOT / "notebooks" / ".demo-memory-06"
shutil.rmtree(memory_root, ignore_errors=True)
checkpoint_manager = CheckpointManager(memory_root)
outbox = RetryableOutbox(memory_root, max_retries=3)

events = [
    JournalEvent(session_id="delivery-demo", mutation_kind="add", changed_files=["active/working-set.jsonl"]),
    JournalEvent(session_id="delivery-demo", mutation_kind="promote", changed_files=["active/claims/extracted.jsonl"]),
]
for event in events:
    outbox.enqueue(event)

show([event.model_dump(mode="json") for event in events])

In [ ]:
attempts = {"count": 0}

def flaky_handler(event):
    attempts["count"] += 1
    return attempts["count"] >= 2

first_pass = outbox.deliver_all(flaky_handler)
checkpoint_manager.save(
    "after-first-pass",
    {
        "attempts": attempts["count"],
        "first_pass": first_pass,
        "pending_outbox_exists": outbox.outbox_file.exists(),
    },
)
second_pass = outbox.deliver_all(lambda event: True)
checkpoint_manager.save(
    "after-second-pass",
    {
        "attempts": attempts["count"],
        "second_pass": second_pass,
        "pending_outbox_exists": outbox.outbox_file.exists(),
    },
)
show({
    "first_pass": first_pass,
    "second_pass": second_pass,
    "checkpoints": checkpoint_manager.list_checkpoints(),
})

In [ ]:
show({
    "after_first_pass": checkpoint_manager.load("after-first-pass"),
    "after_second_pass": checkpoint_manager.load("after-second-pass"),
    "delivered_events_file": str(outbox.delivered_file),
    "delivered_rows": outbox._load_delivered_ids(),
})

## What this notebook demonstrates

- journal events as durable records of change
- retryable delivery for downstream processing
- checkpoint snapshots for resumable agent workflows
- reliability features that make the memory stack operationally useful